In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

WORKSPACE_DIR = "/content/drive/MyDrive/Qwen_SDXL_Trainer"
os.makedirs(WORKSPACE_DIR, exist_ok=True)
%cd {WORKSPACE_DIR}

!nvidia-smi

In [ ]:
import os
from google.colab import userdata

github_token = userdata.get('GITHUB_TOKEN')

REPO_URL_STR = "https://github.com/cosmicoxytocin/qwen-sdxl-adapter.git"
REPO_URL = REPO_URL_STR.replace("https://", f"https://{github_token}@")
REPO_DIR = os.path.join(WORKSPACE_DIR, "qwen-sdxl-adapter")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} -b revert

%cd {REPO_DIR}

!curl -LsSf https://astral.sh/uv/install.sh | sh
import sys
os.environ['PATH'] += f":{os.path.expanduser('~/.cargo/bin')}"
!uv pip install --system -r requirements.txt --quiet

import wandb
from google.colab import userdata
wandb_key = userdata.get('WANDB_API')
if wandb_key:
    os.environ['WANDB_API_KEY'] = wandb_key
    wandb.login(key=wandb_key)


In [ ]:
os.chdir(REPO_DIR)
!uv pip install --system -r requirements.txt

In [ ]:
import os
import glob

DATA_DIR = os.path.join(WORKSPACE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
CACHE_DIR = os.path.join(DATA_DIR, "cache")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"📁 Please ensure your images and .txt files are uploaded to:\n{RAW_DIR}\n")


In [ ]:
import os
import glob

os.chdir(f"{WORKSPACE_DIR}/data/raw")

# Verify
extensions = ("*.jpg", "*.jpeg", "*.png", "*.webp")
image_files = []
for ext in extensions:
    image_files.extend(glob.glob(os.path.join(RAW_DIR, ext)))

if not image_files:
    print("⚠️ No images found! Please upload your files to the directories above and re-run this cell.")
else:
    print(f"✅ Found {len(image_files)} images.")
    missing_txts = 0
    for img_path in image_files:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        txt_path = os.path.join(RAW_DIR, f"{base_name}.txt")

        if os.path.exists(txt_path):
            with open(txt_path, "r", encoding="utf-8") as f:
                prompt = f.read().strip()
            print(f"  - {base_name}: paired with prompt ({len(prompt)} chars)")
        else:
            print(f"  ❌ {base_name}: MISSING .txt FILE!")
            missing_txts += 1
    if missing_txts == 0:
        print("\n🎉 All images have matching text files.")

In [ ]:
#@title AoT Caching
import os

# make sure we're in REPO_DIR
os.chdir(REPO_DIR)

!python scripts/cache_dataset.py \
    --raw_data_dir {RAW_DIR} \
    --cache_dir {CACHE_DIR}

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

In [ ]:
#@title Smoke Test
!python scripts/train.py \
    training.gradient_accumulation_steps=2 \
    training.max_train_steps=1500 \
    training.learning_rate=1e-4 \
    training.lr_warmup_steps=100 \
    training.mixed_precision="bf16" \
    training.save_optimizer_state=False \
    training.checkpointing_steps=500 \
    training.validation_steps=500 \
    data.batch_size=2 \
    data.cache_dir={CACHE_DIR} \
    logging.validation_prompts=["A woman with long dark hair kneels on a white bed. The scene is softly lit with natural light, creating a calm and intimate atmosphere in a minimalist bedroom."] \
    logging.run_name="test-run-316-v2"

In [ ]:
#@title Inference
import IPython.display as display
os.chdir(REPO_DIR)

ckpt_dir = "./checkpoints"
checkpoints = [os.path.join(ckpt_dir, f) for f in os.listdir(ckpt_dir) if f.endswith(".safetensors")]
latest_ckpt = max(checkpoints, key=os.path.getmtime)

print(f"Using checkpoint: {latest_ckpt}")
output_image_path = "./outputs/test_generation.png"

prompt_str = "" #@param {type:'string'}

!python scripts/inference.py \
    --prompt f"{prompt_str}" \
    --adapter_ckpt {latest_ckpt} \
    --steps 30 \
    --cfg 4 \
    --output {output_image_path}

display.Image(output_image_path)